# Ragrails — Retrieval

This notebook covers `retrieve()` — searching a vector database for chunks relevant to a query.

Retrieval uses `input_type="query"` for embedding, which produces a different vector representation than the one used at index time — optimised for search rather than storage.

Optional reranking rescores the top candidates with a cross-encoder to improve precision.

## Install

Install the matching vector database and embedding extras:

```bash
pip install "ragrails[voyage,qdrant]"
pip install "ragrails[voyage,pinecone]"
pip install "ragrails[voyage,weaviate]"
```

For reranking:

```bash
pip install "ragrails[rerank]"
```

In [ ]:
# Qdrant
# %pip install "ragrails[voyage,qdrant]"

# Pinecone
# %pip install "ragrails[voyage,pinecone]"

# Weaviate
# %pip install "ragrails[voyage,weaviate]"

# Reranking
# %pip install "ragrails[rerank]"

## API keys

In [ ]:
import os

os.environ["VOYAGE_API_KEY"] = "your-voyage-api-key"

# Pinecone
# os.environ["PINECONE_API_KEY"] = "your-pinecone-api-key"

# Weaviate Cloud
# os.environ["WEAVIATE_API_KEY"] = "your-weaviate-api-key"

In [ ]:
from ragrails import RagRails

rag = RagRails()

---

## Basic retrieval

`retrieve()` returns the top matching chunks for a query string.

In [ ]:
result = rag.retrieve(
    "How do payouts work?",
    vector_db="qdrant",
    url="http://localhost:6333",
    collection="rag_chunks",
    top_k=10,
)

print("Query:", result.query)
print("Results:", len(result.results))

## Inspect results

Each result is a `RetrievedChunk` with a similarity score, the chunk text, and its metadata.

In [ ]:
for item in result.results:
    print(f"Score: {item.score:.4f}")
    print(f"Title: {item.metadata.get('title')}")
    print(f"Heading: {item.metadata.get('heading')}")
    print(f"Text: {item.text[:300]}")
    print("---")

---

## Retrieval with reranking

Use `rerank=True` to rescore candidates with a cross-encoder. Retrieve more candidates (`top_k`) then narrow to the best (`rerank_top_k`).

In [ ]:
result = rag.retrieve(
    "How do payouts work?",
    vector_db="qdrant",
    url="http://localhost:6333",
    collection="rag_chunks",
    top_k=20,
    rerank=True,
    reranker_model="rerank-2-lite",
    rerank_top_k=5,
)

print("Results after reranking:", len(result.results))

for item in result.results:
    print(f"Score: {item.score:.4f}  Rerank score: {item.rerank_score:.4f}")
    print(f"Text: {item.text[:200]}")
    print("---")

---

## Retrieval with Pinecone

In [ ]:
result = rag.retrieve(
    "How do payouts work?",
    vector_db="pinecone",
    collection="rag-chunks",  # hyphens only, no underscores
    top_k=10,
)

for item in result.results:
    print(item.score, item.text[:200])

---

## Retrieval with Weaviate

In [ ]:
result = rag.retrieve(
    "How do payouts work?",
    vector_db="weaviate",
    url="http://localhost:8080",
    collection="RagChunks",
    top_k=10,
)

for item in result.results:
    print(item.score, item.text[:200])

---

## Result type reference

```python
RetrieveResult(
    query=str,
    results=[
        RetrievedChunk(
            id=str,
            score=float,
            text=str,
            metadata=dict,
            rerank_score=float | None,
        )
    ],
)
```

`rerank_score` is `None` when reranking is not used.

---

## Error handling

In [ ]:
try:
    result = rag.retrieve(
        "How do payouts work?",
        vector_db="qdrant",
        collection="rag_chunks",
    )
except ValueError as e:
    print("Validation error:", e)
except RuntimeError as e:
    # Missing optional dependency
    print("Setup error:", e)

### Common failure causes

| Provider | Common cause |
|---|---|
| All | `VOYAGE_API_KEY` missing or invalid |
| All | Collection does not exist — run `embed()` first |
| Qdrant | Qdrant not running, or port 6333 not exposed |
| Pinecone | `PINECONE_API_KEY` missing, or collection name has underscores |
| Weaviate | gRPC port 50051 not exposed, or collection name not uppercase |
| Reranking | `ragrails[rerank]` not installed |